In [ ]:
# Install necessary libraries
!pip install gradio diffusers torch transformers accelerate safetensors python-dotenv
!pip install pillow requests

In [9]:
import gradio as gr
import torch
import requests
import os
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline

from google.colab import userdata
# Hugging Face API Key
API_TOKEN = userdata.get('H')

# Whisper API Endpoint
WHISPER_API_URL = "https://api-inference.huggingface.co/models/openai/whisper-small"
HEADERS = {"Authorization": f"Bearer {API_TOKEN}"}

# Load Stable Diffusion img2img pipeline
def load_pipeline():
    """
    Load the Stable Diffusion img2img pipeline.
    """
    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16
    )
    pipe.to("cuda")  # Use GPU for faster processing
    return pipe

# Load the pipeline once (first run may take time)
pipe = load_pipeline()

# Function to transcribe audio
def transcribe_audio(audio_filepath):
    """
    Transcribes an audio file to text using the Whisper API.
    """
    try:
        with open(audio_filepath, "rb") as f:
            response = requests.post(WHISPER_API_URL, headers=HEADERS, files={"file": f})

        if response.status_code == 200:
            result = response.json()
            return result.get("text", "No transcription available")
        else:
            return f"Error: {response.status_code}, {response.text}"

    except Exception as e:
        return str(e)

# Function to modify the image based on the transcribed text
def generate_image(input_image, audio_filepath):
    """
    Given an input image and an audio file, transcribes the audio into text
    and uses the text as a prompt to modify the input image using Stable Diffusion img2img.

    Returns both the transcription and the modified image.
    """
    if input_image is None:
        return "No image provided.", None

    # Get the transcription
    transcription = transcribe_audio(audio_filepath)
    if transcription.startswith(("Error", "Audio conversion error", "Transcription failed")):
        return transcription, None

    prompt = transcription  # Use the transcribed text as the prompt
    print("Transcription:", prompt)  # Debug output

    # Convert image to RGB and resize to 512x512
    image = input_image.convert("RGB").resize((512, 512))

    # Generate the modified image using Stable Diffusion
    result = pipe(
        prompt=prompt,
        image=image,
        strength=0.75,         # Controls how strongly the prompt is applied
        num_inference_steps=50, # Number of denoising steps
        guidance_scale=7.5,
    ).images[0]

    return f"Prompt taken from audio file/speech: {prompt}", result

# Gradio Interface
iface = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Image(type="pil", label="Upload an Image"),  # User uploads an image
        gr.Audio(type="filepath", label="Upload an Audio File")  # User uploads an audio file
    ],
    outputs=[
        gr.Textbox(label="Prompt taken from audio file/speech"),  # Display transcribed text with a clear label
        gr.Image(type="pil", label="Modified Image")  # Display the modified image
    ],
    title="Audio-Powered Image Modification",
    description="Upload an audio file and an image. The system will transcribe the audio and use the text to modify the image using Stable Diffusion."
)

# Launch the interface
iface.launch(share=True)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://412735f4645f2c08b1.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
